06/03/2026

Mik va a intentar hacer una red convolucional cn pytorch, lol

Estoy utilizando el env dl2024 

- Cloth**Dataset** guarda la info d vertices x caracteristicas () al acceder a estos items con el **DataLoader** le añade la otra dimension d frames(batchsize) para crear el tensor3D
- Redondear valores para optimizar (ahorra memoria)

Ahora el modelo itera con distintos batchSizes en modo shuffle, para ir entrenandose poco a poco. No tiene memoria, pero como guardamos las velocidades y tal probablemente funcione?
> Your model assumes that the current state of the cloth is all it needs to predict the next state (this is called a Markov assumption). In this setup, the network looks at a single frame's positions and velocities and predicts the displacements. Graph Neural Networks (GNNs) or standard Multi-Layer Perceptrons (MLPs) usually take data in this exact shape.

Otra idea sería:
> When to add a frame dimension (Sequence modeling): If your model needs temporal history—meaning it needs to look at, say, the last 5 frames to figure out what happens in the 6th frame. If you were using an LSTM, RNN, or a Spatiotemporal Transformer, your tensor would need to look like [Batch, Sequence_Length, Vertices, Features].
Por ahora no.

**Links Utilizados:**
- https://docs.pytorch.org/tutorials/beginner/data_loading_tutorial.html
- https://lixiaoguang.medium.com/build-cnn-from-scratch-5-convolutional-neural-network-86b4d0323fb0

In [2]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import io
import torch
import os, os.path

# WORKING WITH 
datasetPath = 'data/'

def loadAndMergeCSV(csvRoute):
    """
    Carga de los CSV y mergeo en un único CSV. Todos los CSV estarán en la ruta 'data/', y se excluirá el CSV
    'mergedCSV.csv', producto de los mergeos si se ejecutase antes
    """
    csvRoute= 'data/'
    finalData = pd.DataFrame()
    for csvfile in [f for f in os.listdir(csvRoute) if os.path.isfile(csvRoute + f)]:
        if (csvfile != 'mergedCSV.csv' and os.path.splitext(csvfile)[1] == '.csv'):
            data = pd.read_csv(csvRoute + csvfile)
            finalData = pd.concat([data, finalData], ignore_index=True)

    return finalData

cloth_info = loadAndMergeCSV(datasetPath)

print('cloth_info shape: {}'.format(cloth_info.shape))
print('cloth_info: \n{}'.format(cloth_info))

cloth_info shape: (468, 79)
cloth_info: 
     frame            x0        y0        z0        vx0       vy0       vz0  \
0        0 -4.216044e-08  1.499913  0.225583  -0.004361 -74.99863  0.000992   
1        1  1.820105e-02  1.500293  0.219883  -0.886701 -75.92898 -0.049198   
2        2 -1.066346e-01  1.511619  0.222103   5.462661 -70.14699  0.015023   
3        3 -1.996845e-01  1.541488  0.224504  12.126770 -67.12998 -0.001404   
4        4 -6.189827e-03  1.500170  0.222646   1.266535 -74.72750  0.020391   
..     ...           ...       ...       ...        ...       ...       ...   
463    463 -1.047743e-01  1.511029  0.223596   6.366400 -70.44193  0.041204   
464    464 -2.453499e-02  1.500698  0.224925   1.329836 -73.80412 -0.014795   
465    465 -7.667037e-02  1.505977  0.221374   3.542761 -71.37963  0.058286   
466    466 -2.373345e-01  1.559896  0.218972  14.427990 -65.90613 -0.060602   
467    467 -1.515977e-01  1.523507  0.225559   9.442355 -68.82992  0.005285   

         s

In [4]:
class ClothDataset(Dataset):
    def __init__(self, csv_data, num_vertices=6):
        """
        Args:
            csv_data (str or filepath): Path to the CSV file or raw CSV string.
            num_vertices (int): Number of vertices per frame.
        """
        # Load the CSV data into a pandas DataFrame
        #if isinstance(csv_data, str) and "frame,x0" in csv_data:
            #self.data = pd.read_csv(io.StringIO(csv_data.strip()))
        #else:
            #self.data = pd.read_csv(csv_data)
        self.data = csv_data
            
        #quick fix para espacios en primera fila
        self.data.columns = self.data.columns.str.strip()
        
        self.num_vertices = num_vertices
        
        # Define the base feature names to extract per vertex
        self.feature_prefixes = ['x', 'y', 'z', 'sdf'] # 'vx', 'vy', 'vz',, 'nx', 'ny', 'nz', 'md', 'u', 'v'

        self.position_prefixes = ['x', 'y', 'z']
        self.output_positions = self.data.filter(regex=r'^[xyz]\d+$')

        # Quitamos primera fila de outputs (no es el output de nada) y ultima fila de input (no tiene output)
        self.output_positions = self.output_positions.iloc[1:]
        self.data = self.data.iloc[:-1]
        

    def __len__(self):
        # The number of items is the number of frames (rows) in the dataset
        return len(self.data)
    
    def num_features(self):
        # Number of columns in a row (pos, vel, sdf, uv per vertex)
        return len(self.feature_prefixes)
    
    def num_vertex(self):
        return self.num_vertices
    
    def _get_frame_output_tensor(self, idx):
         row = self.output_positions.iloc[idx]
         frame_data = []
         for i in range(self.num_vertices):
            vertex_cols = [f"{prefix}{i}" for prefix in self.position_prefixes]
            vertex_features = row[vertex_cols].values.astype(np.float32)
            frame_data.append(vertex_features)

         tensor_data = torch.tensor(np.array(frame_data))

         return tensor_data
    
    def _get_frame_tensor(self, idx):
        row = self.data.iloc[idx]
        frame_data = []

        for i in range(self.num_vertices):
            vertex_cols = [f"{prefix}{i}" for prefix in self.feature_prefixes]
            vertex_features = row[vertex_cols].values.astype(np.float32)
            frame_data.append(vertex_features)

        tensor_data = torch.tensor(np.array(frame_data))

        return tensor_data
        
    def __getitem__(self, idx):
        frame_t = self._get_frame_tensor(idx)
        frame_t1 = self._get_frame_output_tensor(idx) # +1 ya no TODO: Pillar solo las columnas de pos

        return frame_t, frame_t1

# --- Example Usage ---

# (Assuming 'csv_string' is a variable holding your provided data block)
dataset = ClothDataset(cloth_info)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

for batch_data, batch_frames in dataloader:
    print(f"Batch Shape: {batch_data.shape}")
    print(batch_data)
    print(batch_frames)
    break


mean = 0
std = 0
n_samples = 0

for batch_t, _ in dataloader:
    batch_t = batch_t.float()
    
    batch_samples = batch_t.size(0)
    batch_t = batch_t.view(-1, batch_t.size(-1))  

    mean += batch_t.mean(dim=0)
    std += batch_t.std(dim=0)
    n_samples +=  batch_t.size(0)

mean /= n_samples
std /= n_samples

# evitamos division por 0
std[std < 1e-8] = 1.0

print("MEAN:", mean)
print("STD:", std)

Batch Shape: torch.Size([4, 6, 4])
tensor([[[ 4.0908e-02,  1.5020e+00,  2.0997e-01,  1.5074e-01],
         [-2.4015e-01,  1.0288e+00, -1.6381e-01, -3.9316e-01],
         [ 6.4461e-02,  9.9932e-01,  1.6749e-01, -7.8522e-02],
         [-1.1799e-01,  1.5143e+00, -2.1167e-01, -1.5063e-02],
         [-5.9605e-08,  2.0000e+00,  2.2550e-01,  5.6838e-01],
         [-5.9605e-08,  2.0000e+00, -2.2550e-01,  4.8033e-01]],

        [[-3.2256e-02,  1.5011e+00,  2.2027e-01,  2.2853e-01],
         [ 3.8175e-01,  1.1155e+00, -1.3815e-01, -3.8306e-01],
         [ 1.8447e-01,  1.0490e+00,  2.6179e-01, -1.5057e-02],
         [ 7.2193e-02,  1.5052e+00, -2.1831e-01,  4.1576e-02],
         [-5.9605e-08,  2.0000e+00,  2.2550e-01,  6.0982e-01],
         [-5.9605e-08,  2.0000e+00, -2.2550e-01,  5.2533e-01]],

        [[ 2.5536e-01,  1.5703e+00,  2.2303e-01,  3.6122e-01],
         [ 2.5377e-01,  1.0374e+00, -1.8527e-01,  2.2303e-02],
         [ 4.0508e-01,  1.0920e+00,  2.3594e-01,  7.5614e-02],
         [ 1.866

In [5]:
import json
# Intento de normalización uep

# Convertir tensores a listas
norm_data = {
    "mean": mean.tolist(),
    "std": std.tolist(),
    "feature_prefixes": ['x', 'y', 'z',  'sdf']# 'vx', 'vy', 'vz',, 'nx', 'ny', 'nz', 'md', 'u', 'v'
}

with open("cloth_norm_params.json", "w") as f:
    json.dump(norm_data, f)

In [6]:
# TODO
# una recurrente sencilla (la salida se vuelve entrada en el siguiente ejemplo)
# Antes de meternos en CNN y LSTM
# AÑADIMOS VALORES U V PARA CADA VERTICE ( no queremos perder la noción espacial )

import torch.nn as nn
import torch.nn.functional as F #acceso rapido a funciones
import torch.utils.data as data #cargar y manejar el training data

class MyModule(nn.Module):

    def __init__(self, num_inputs, num_hidden, num_outputs):
        super().__init__()
        # Some init for my module
        self.linear1 = nn.Linear(num_inputs, num_hidden)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(num_hidden, num_outputs)

    def forward(self, x):
        # Function for performing the calculation of the module.
        
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        #print(x)
        return x

    #backward se hace automaticamente, podriamos definirla tmbn 

#tmbn clases DataSet y DataLoader

# definir modelo, loss function y optimizer
#TODO: buscar dimensiones reales de las neuronas

# nn.Linear in PyTorch is designed to handle 3D tensors seamlessly.  
# When a 3D input tensor (e.g., batch_size, sequence_length, features) is provided,
#  the layer applies the linear transformation only to the last dimension (the features dimension),
#  preserving all other dimensions.

model = MyModule(num_inputs=4, num_hidden= 64, num_outputs=3)
# print, save, lo que sea
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

for epoch in range(100):
    for batch_t, batch_t1 in dataloader:

        batch_t = batch_t.float()
        batch_t1 = batch_t1.float()

        batch_t = (batch_t - mean) / std

        # output (posiciones)
        mean_out = batch_t1.mean(dim=(0,1), keepdim=True)
        std_out = batch_t1.std(dim=(0,1), keepdim=True) + 1e-8
        batch_t1 = (batch_t1 - mean_out) / std_out

        pred = model(batch_t)

        loss = criterion(pred, batch_t1)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        print(f'Epoch {epoch+1}, Loss: {loss.item()}')


Epoch 1, Loss: 147.79412841796875
Epoch 1, Loss: 138.6722869873047
Epoch 1, Loss: 96.37046813964844
Epoch 1, Loss: 149.29800415039062
Epoch 1, Loss: 146.8673095703125
Epoch 1, Loss: 138.57737731933594
Epoch 1, Loss: 177.4570770263672
Epoch 1, Loss: 112.89289093017578
Epoch 1, Loss: 168.9864044189453
Epoch 1, Loss: 119.26947784423828
Epoch 1, Loss: 89.4556884765625
Epoch 1, Loss: 150.4027862548828
Epoch 1, Loss: 108.48377990722656
Epoch 1, Loss: 111.8314437866211
Epoch 1, Loss: 136.98193359375
Epoch 1, Loss: 143.5851287841797
Epoch 1, Loss: 115.67831420898438
Epoch 1, Loss: 128.4371337890625
Epoch 1, Loss: 147.5249481201172
Epoch 1, Loss: 117.46585845947266
Epoch 1, Loss: 109.98248291015625
Epoch 1, Loss: 96.30268096923828
Epoch 1, Loss: 93.63399505615234
Epoch 1, Loss: 115.55889129638672
Epoch 1, Loss: 99.57649230957031
Epoch 1, Loss: 111.37194061279297
Epoch 1, Loss: 99.09971618652344
Epoch 1, Loss: 76.77336120605469
Epoch 1, Loss: 86.70114135742188
Epoch 1, Loss: 138.3889923095703
Ep

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import numpy as np

# ... [Keep your ClothDataset and MyModule classes exactly as they are] ...

dataset = ClothDataset(cloth_info)

# 1. SPLIT THE DATA (80% Training, 20% Testing)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False) # No need to shuffle test data

# 2. COMPUTE GLOBAL STATS (Only on training data to prevent data leakage)
mean_in = 0
std_in = 0
n_samples = 0

for batch_t, _ in train_loader:
    batch_t = batch_t.float()
    batch_t_flat = batch_t.view(-1, batch_t.size(-1))  
    mean_in += batch_t_flat.mean(dim=0)
    std_in += batch_t_flat.std(dim=0)
    n_samples += batch_t.size(0)

global_in_mean = mean_in / n_samples
global_in_std = std_in / n_samples
global_in_std[global_in_std < 1e-8] = 1.0

# 3. SETUP MODEL
model = MyModule(num_inputs=4, num_hidden=64, num_outputs=3)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

print("start training")

# 4. TRAINING & TESTING LOOP
epochs = 100
for epoch in range(epochs):
    
    # --- TRAINING PHASE ---
    model.train() # Set model to training mode
    train_loss = 0.0
    
    for batch_t, batch_t1 in train_loader:
        batch_t = batch_t.float()
        batch_t1 = batch_t1.float()

        # Normalize Input globally
        batch_t_norm = (batch_t - global_in_mean) / global_in_std

        # Target is the DISPLACEMENT (Delta), not the absolute position
        positions_t = batch_t[..., 0:3] 
        target_delta = batch_t1 - positions_t
        
        # Note: For best results, target_delta should also be globally normalized, 
        # but we will keep it raw here for easier "Distance Error" calculation later.

        pred_delta = model(batch_t_norm)
        loss = criterion(pred_delta, target_delta)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        train_loss += loss.item()
        
    print(f'Train Epoch {epoch+1:03d}')
        
    avg_train_loss = train_loss / len(train_loader)

    # --- TESTING/VALIDATION PHASE ---
    model.eval() # Set model to evaluation mode
    test_loss = 0.0
    total_distance_error = 0.0
    total_vertices = 0
    
    with torch.no_grad(): # Disable gradient calculation for testing (saves memory & speeds up)
        for batch_t, batch_t1 in test_loader:
            batch_t = batch_t.float()
            batch_t1 = batch_t1.float()

            batch_t_norm = (batch_t - global_in_mean) / global_in_std
            
            positions_t = batch_t[..., 0:3] 
            target_delta = batch_t1 - positions_t

            pred_delta = model(batch_t_norm)
            loss = criterion(pred_delta, target_delta)
            test_loss += loss.item()
            
            # Calculate our "Accuracy" (Mean Distance Error per vertex)
            # Distance = sqrt((dx_pred - dx_true)^2 + (dy_pred - dy_true)^2 + (dz_pred - dz_true)^2)
            distances = torch.norm(pred_delta - target_delta, dim=-1) # Calculates Euclidean distance
            total_distance_error += distances.sum().item()
            total_vertices += distances.numel()

            print(f'Epoch {epoch+1:03d} | Test Loss: {test_loss:.6f} | Vertex Error: {total_distance_error:.6f} units')

    avg_test_loss = test_loss / len(test_loader)
    avg_distance_error = total_distance_error / total_vertices

    print(f'Epoch {epoch+1:03d} | Train Loss: {avg_train_loss:.6f} | Test Loss: {avg_test_loss:.6f} | Avg Vertex Error: {avg_distance_error:.6f} units')

start training
Train Epoch 001
Epoch 001 | Test Loss: 0.031801 | Vertex Error: 6.660538 units
Epoch 001 | Test Loss: 0.065981 | Vertex Error: 13.885348 units
Epoch 001 | Test Loss: 0.091438 | Vertex Error: 19.958813 units
Epoch 001 | Test Loss: 0.119596 | Vertex Error: 26.557384 units
Epoch 001 | Test Loss: 0.154375 | Vertex Error: 33.274130 units
Epoch 001 | Test Loss: 0.182479 | Vertex Error: 39.794626 units
Epoch 001 | Test Loss: 0.220173 | Vertex Error: 47.138237 units
Epoch 001 | Test Loss: 0.252825 | Vertex Error: 53.555266 units
Epoch 001 | Test Loss: 0.277128 | Vertex Error: 59.614011 units
Epoch 001 | Test Loss: 0.303289 | Vertex Error: 66.043761 units
Epoch 001 | Test Loss: 0.333044 | Vertex Error: 72.729825 units
Epoch 001 | Test Loss: 0.356391 | Vertex Error: 78.715604 units
Epoch 001 | Test Loss: 0.388162 | Vertex Error: 85.505367 units
Epoch 001 | Test Loss: 0.419092 | Vertex Error: 92.072971 units
Epoch 001 | Test Loss: 0.435453 | Vertex Error: 97.053569 units
Epoch 001 

In [11]:
#INTENTO DE EXPORTAR A ONNX
import sys
print(sys.executable)

import onnx
import onnxruntime

print("ONNX version:", onnx.__version__)
print("ONNX Runtime version:", onnxruntime.__version__)

# Create example inputs for exporting the model. The inputs should be a tuple of tensors.
# example_inputs = (batch_t)
example_inputs = batch_t[0:1, :, :]
onnx_program = torch.onnx.export(model, example_inputs, dynamo=True)

onnx_program.save("posBased6Train.onnx")

c:\Users\mikel\miniconda3\envs\dl2024\python.exe
ONNX version: 1.21.0
ONNX Runtime version: 1.24.4
[torch.onnx] Obtain model graph for `MyModule([...]` with `torch.export.export`...
[torch.onnx] Obtain model graph for `MyModule([...]` with `torch.export.export`... ✅
[torch.onnx] Translate the graph into ONNX...


W0420 13:31:41.771000 21956 site-packages\torch\onnx\_internal\exporter\_schemas.py:446] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0420 13:31:41.773000 21956 site-packages\torch\onnx\_internal\exporter\_schemas.py:446] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0420 13:31:41.775000 21956 site-packages\torch\onnx\_internal\exporter\_schemas.py:446] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0420 13:31:41.777000 21956 site-packages\torch\onnx\_internal\exporter\_schemas.py:446] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 

[torch.onnx] Translate the graph into ONNX... ✅


In [26]:
import torch
import torch.nn as nn

#EJEMPLO SENCILLO CONVOLUCIONAL PARA MÁS ADELANTE

# Example: 100 features, 1 channel (linear input)
# Batch size=16
input_data = torch.randn(16, 1, 100) 

model = nn.Sequential(
    nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3), # Extract features
    nn.ReLU(),
    nn.Flatten(), # Flatten for Dense layer
    nn.Linear(32 * 98, 10) # 98 is the new length after convolution
)
